# Stuttering Detection: Probabilistic Classification Analysis
**Course**: CS204T (Artificial Intelligence)  
**Team**: 18  
**Focus**: Naive Bayes vs. Normal Bayes (LDA)

---

## Step 1: Initialization & Environment
We use our modular project engine to load the 768-dimensional WavLM embeddings and prepare a fair, balanced experimental setup.

In [1]:
import os
import shutil
import numpy as np
from src.extractors import WavLMExtractor
from src.data import DataManager
from src.models import NaiveBayesModel, LDAModel

# Dataset Configuration
AUDIO_DIR = "Stuttering Events in Podcasts Dataset/clips/stuttering-clips/clips"
CSV_PATHS = [
    "Stuttering Events in Podcasts Dataset/SEP-28k_labels.csv",
    "Stuttering Events in Podcasts Dataset/fluencybank_labels.csv"
]
FEATURE_DIR = "data/features"
fluent_dir = os.path.join(FEATURE_DIR, "fluent")
disfluent_dir = os.path.join(FEATURE_DIR, "disfluent")

/home/anshuman139/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Step 2 (Optional): Operational Mode for Data Extraction
Toggle how you want to handle your audio data for this session.
* `SKIP_EXTRACTION`: Uses features already on disk (Default).
* `FORCE_EXTRACT`: Analyzes raw audio for new files (Resumable).
* `CLEAN_START`: Wipes the database and re-extracts from zero.

In [ ]:
# Operational Flags
SKIP_EXTRACTION = True
FORCE_EXTRACT = False
CLEAN_START = False
NUM_CLIPS_TO_EXTRACT = 5

if CLEAN_START:
    if os.path.exists(FEATURE_DIR):
        shutil.rmtree(FEATURE_DIR)
    print("[System] Clean start initiated. Feature database wiped.")

if not SKIP_EXTRACTION or CLEAN_START or FORCE_EXTRACT:
    extractor = WavLMExtractor("microsoft/wavlm-base")
    label_dict = DataManager.generate_label_dict(CSV_PATHS, filter_quality=True)
    sample_wavs = [os.path.join(AUDIO_DIR, f) for f in os.listdir(AUDIO_DIR) 
                   if f.lower().endswith('.wav')][:NUM_CLIPS_TO_EXTRACT]
    extractor.extract_batch(sample_wavs, output_dir=FEATURE_DIR, label_dict=label_dict)
else:
    print("[System] Skipping extraction. Using existing data on disk.")

## Step 3: Data Preparation Engine
We load our distributed features, filter for quality, and use **Oversampling** to handle the 1:3 class imbalance between fluent and disfluent speech.

In [2]:
label_dict = DataManager.generate_label_dict(CSV_PATHS, filter_quality=True)
manager = DataManager(None, None)

# Loading .npy features into matrices
X, y = manager.load_from_folders(fluent_dir, disfluent_dir)
X_train, X_val, X_test, y_train, y_val, y_test = manager.get_splits(test_size=0.15, val_size=0.15)

# Handle Class Imbalance
X_train_bal, y_train_bal = manager.balance_data(X_train, y_train, strategy="oversample")

# Standard Scaling (Fitted on train only)
X_train_final = manager.preprocess(X_train_bal, method="standard")
X_test_final = manager.preprocess(X_test, method="standard")

[DataManager] Quality Filter: Removed 3938 low-quality samples.


## Step 4: Model 1 - Naive Bayes Classifier
The Naive Bayes model applies Bayes' Theorem with the strong assumption that every feature is independent. Interestingly, this 'simple' approach often results in the **highest Recall** for stuttering detection.

In [3]:
nb_model = NaiveBayesModel("Naive_Bayes_Baseline")
nb_model.train(X_train_final, y_train_bal)

print("\n--- Evaluation on Unseen Test Set ---")
nb_model.evaluate(X_test_final, y_test)

[Model: Naive_Bayes_Baseline] Initialized.
[Naive_Bayes_Baseline] Training on 1778 samples...

--- Evaluation on Unseen Test Set ---

--- Evaluation: Naive_Bayes_Baseline ---
Accuracy: 0.6174
Precision: 0.4762
Recall: 0.6542
F1: 0.5512

Confusion Matrix (Binary):
               Predicted: Fluent(0)  Predicted: Stutter(1)
True: Fluent(0)      114             77             
True: Stutter(1)     37              70             


{'accuracy': 0.6174496644295302,
 'precision': 0.47619047619047616,
 'recall': 0.6542056074766355,
 'f1': 0.5511811023622047,
 'confusion_matrix': array([[114,  77],
        [ 37,  70]])}

## Step 5: Model 2 - Normal Bayes (LDA)
Linear Discriminant Analysis (LDA) is often referred to as 'Normal Bayes.' Unlike the Naive version, it accounts for the covariance between features, attempting to model the entire dataset as a multi-dimensional Gaussian distribution.

In [4]:
lda_model = LDAModel("Normal_Bayes_LDA")
lda_model.train(X_train_final, y_train_bal)

print("\n--- Evaluation on Unseen Test Set ---")
lda_model.evaluate(X_test_final, y_test)

[Model: Normal_Bayes_LDA] Initialized.
[Normal_Bayes_LDA] Training Bayesian Probability Map...

--- Evaluation on Unseen Test Set ---

--- Evaluation: Normal_Bayes_LDA ---
Accuracy: 0.5638
Precision: 0.4207
Recall: 0.5701
F1: 0.4841

Confusion Matrix (Binary):
               Predicted: Fluent(0)  Predicted: Stutter(1)
True: Fluent(0)      107             84             
True: Stutter(1)     46              61             


{'accuracy': 0.5637583892617449,
 'precision': 0.4206896551724138,
 'recall': 0.5700934579439252,
 'f1': 0.48412698412698413,
 'confusion_matrix': array([[107,  84],
        [ 46,  61]])}

## Step 6: Comparative Results
By analyzing the side-by-side Confusion Matrices, we reach a fascinating conclusion:

**Finding**: Although LDA is statistically more complex, the **Naive Bayes** model consistently catches more stutters (Higher Recall). This suggests that the 'Independence Assumption' actually helps the model ignore the high-dimensional noise found in 768-dimensional audio embeddings.

In [7]:
X_train_final[0][len(X_train_final[0])-1]

np.float32(-0.9462235)